## Importações

In [3]:
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import dateparser
from datetime import datetime, timedelta
import re

## Caminhos dos arquivos html

In [ ]:
BASE_PATHS = [
    "../database/Nacarelli/Takeout/YouTube e YouTube Music",
    "../database/Giovanna/Takeout/YouTube e YouTube Music",
    "../database/Leticia/Takeout/YouTube e YouTube Music"
]

## Função que padroniza os dados que alimentarão a árvore de decisão

- Validação de links;
- Padroniza a data
- Transforma em um DataFrame com o título do vídeo e a data

In [5]:
def gerar_dataset_modelo(base_path):
    base_dir = Path(base_path) / "histórico"
    arquivo = base_dir / "histórico-de-visualização.html"

    with open(arquivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    dados = []

    blocos = soup.find_all(
        "div",
        class_="content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1"
    )

    for bloco in blocos:
        link_tag = bloco.find("a")
        if not link_tag:
            continue

        url = link_tag["href"]
        texto = bloco.get_text(separator=" ", strip=True)

        padrao_data = r"\d{1,2} de .*? de \d{4}, \d{2}:\d{2}:\d{2}"
        match = re.search(padrao_data, texto)
        if match:
            data_str = match.group()
            data_extraida = dateparser.parse(
                data_str,
                languages=["pt"],
                settings={"TIMEZONE": "America/Sao_Paulo"}
            )
        else:
            continue

        # Exemplo de extração de retenção (ajuste conforme estrutura real do HTML)
        padrao_reten = r"Retenção: (\d+)s"
        match_reten = re.search(padrao_reten, texto)
        retencao = int(match_reten.group(1)) if match_reten else None

        if not data_extraida:
            continue

        tipo = texto.split(link_tag.text.strip())[0].strip()
        if "watched" not in tipo.lower():
            continue

        # Filtrar cliques acidentais: ignorar vídeos com retenção de até 10s
        if retencao is not None and retencao <= 10:
            continue

        dados.append([url, data_extraida, retencao])

    df = pd.DataFrame(dados, columns=["titulo", "data", "retencao"])
    return df


## Criação do csv

- Pega os dados dos últimos 30 dias;
- Coluna "proximo_video" baseada nos vídeos anteriores ao visto;
- Elimina todos os valores nulos;
- Cria um csv com o vídeo assistido, o próximo vídeo e a data

In [ ]:
dfs = []

for path in BASE_PATHS:
    df_temp = gerar_dataset_modelo(path)
    dfs.append(df_temp)

# Concatenar todos
df_final = pd.concat(dfs, ignore_index=True)

limite = datetime.now() - timedelta(days=15)
df_final = df_final[df_final["data"] >= limite]

df_final = df_final.sort_values("data")

df_final["proximo_video"] = df_final["titulo"].shift(-1)
df_final["retencao_proximo"] = df_final["retencao"].shift(-1)

df_final = df_final.dropna()

print("Total de registros:", df_final.shape)

df_final.to_csv("df_modelo_30dias.csv", index=False)
